In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_openrouter import ChatOpenRouter
from typing import TypedDict ,Literal
from dotenv import load_dotenv
from pydantic import BaseModel,Field

c:\Users\Admin\LangGraph2.0\langgraphenv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenRouter(model ="meta-llama/llama-3.1-8b-instruct")


In [4]:
class SentimentSchema(BaseModel):

    sentiment:Literal["positive","negative"]=Field(description='Sentiment of the review')
    

In [5]:

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [6]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model1 = model.with_structured_output(DiagnosisSchema)

In [19]:
class ReviewState(TypedDict):

    review:str
    sentiment:Literal["positive","negative"]
    diagnosis:dict
    response:str

In [20]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
    
def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    
    response = model.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:

{state['review']}

Return issue_type, tone, and urgency.
"""

    response = structured_model1.invoke(prompt)

    return {
        "diagnosis": response.model_dump()
    }


def negative_response(state: ReviewState):

    diagnosis = state["diagnosis"]

    prompt = f"""You are a support assistant.

The user had a '{diagnosis["issue_type"]}' issue,
sounded '{diagnosis["tone"]}',
and marked urgency as '{diagnosis["urgency"]}'.

Write an empathetic, helpful resolution message.
"""

    response = model.invoke(prompt).content

    return {
        "response": response
    }

In [21]:

graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [22]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'},
 'response': 'Here\'s a possible response:\n\n"Hi there, I\'m so sorry to hear that you\'re experiencing the issue and I can sense your frustration! I can see that you\'ve marked the urgency level as high and I\'m here to help resolve it asap.\n\nCan you please tell me a bit more about the problem you\'re facing? What\'s happening with the [insert issue here] and how are you trying to [insert task/action]? I\'m all ears and will do my best to get to the bottom of this and find a solution for you as quickly as possible."'}